In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("ner_dataset.csv", encoding="latin1")
df["Sentence #"] = df["Sentence #"].ffill()
df = df.dropna(subset=["Word", "POS"])

df["Word"] = df["Word"].astype(str)
df["POS"] = df["POS"].astype(str)
grouped = df.groupby("Sentence #")

sentences = grouped["Word"].apply(list).tolist()
tags = grouped["POS"].apply(list).tolist()

print("Total Sentences:", len(sentences))

X_train, X_test, y_train, y_test = train_test_split(
    sentences,
    tags,
    test_size=0.2,
    random_state=42
)
tag_set = set()

for tag_seq in y_train:
    tag_set.update(tag_seq)
tags_list = sorted(list(tag_set))

tag2idx = {tag: idx for idx, tag in enumerate(tags_list)}
idx2tag = {idx: tag for tag, idx in tag2idx.items()}

n_tags = len(tags_list)

print("Number of POS Tags:", n_tags)
initial_counts = np.ones(n_tags)
transition_counts = np.ones((n_tags, n_tags))
emission_counts = defaultdict(Counter)
tag_totals = Counter()

for sentence, tag_seq in zip(X_train, y_train):

    initial_counts[tag2idx[tag_seq[0]]] += 1

    for i in range(len(sentence)):

        word = sentence[i]
        tag = tag_seq[i]

        emission_counts[tag][word] += 1
        tag_totals[tag] += 1

        if i > 0:

            prev_tag = tag_seq[i - 1]
            curr_tag = tag

            transition_counts[
                tag2idx[prev_tag],
                tag2idx[curr_tag]
            ] += 1

initial_prob = initial_counts / initial_counts.sum()

transition_prob = (
    transition_counts
    / transition_counts.sum(axis=1, keepdims=True)
)

log_initial = np.log(initial_prob)
log_transition = np.log(transition_prob)

# Vocabulary size
all_words = set()

for sent in X_train:
    all_words.update(sent)

vocab_size = len(all_words)

print("Vocabulary Size:", vocab_size)

def get_log_emission(word):

    emission = np.zeros(n_tags)

    for i, tag in enumerate(tags_list):
        count = emission_counts[tag][word]
        prob = (
            count + 1
        ) / (
            tag_totals[tag] + vocab_size
        )
        emission[i] = np.log(prob)
    return emission
def viterbi(sentence):

    T = len(sentence)

    dp = np.full((n_tags, T), -np.inf)

    backpointer = np.zeros((n_tags, T), dtype=int)
    emission = get_log_emission(sentence[0])

    dp[:, 0] = log_initial + emission
    for t in range(1, T):
        emission = get_log_emission(sentence[t])
        scores = (
            dp[:, t - 1].reshape(-1, 1)
            + log_transition
        )

        backpointer[:, t] = np.argmax(
            scores,
            axis=0
        )

        dp[:, t] = (
            np.max(scores, axis=0)
            + emission
        )

    # Backtracking
    best_last = np.argmax(dp[:, -1])

    best_path = [best_last]

    for t in range(T - 1, 0, -1):

        best_last = backpointer[
            best_last,
            t
        ]

        best_path.append(best_last)

    best_path.reverse()

    return [
        idx2tag[state]
        for state in best_path
    ]

print("Predicting...")

y_true = []
y_pred = []

for sentence, true_tags in zip(X_test, y_test):

    pred_tags = viterbi(sentence)

    y_true.extend(true_tags)
    y_pred.extend(pred_tags)
accuracy = accuracy_score(
    y_true,
    y_pred
)

print("\n" + "=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        zero_division=0
    )
)
unseen_sentences = [

    "I love machine learning",

    "The cat sat on the mat",

    "Artificial intelligence is amazing",

    "She writes beautiful stories",

    "Students are studying hard"
]

print("\n" + "=" * 60)
print("UNSEEN SENTENCE TESTING")
print("=" * 60)

for sentence in unseen_sentences:

    words = sentence.split()

    pred_tags = viterbi(words)

    print("\nSentence:")
    print(sentence)
    print("\nPredicted Tags:")
    for word, tag in zip(words, pred_tags):

        print(f"{word:15} --> {tag}")
while True:

    text = input(
        "\nEnter sentence (type quit to stop): "
    )

    if text.lower() == "quit":
        break
    words = text.split()
    tags_pred = viterbi(words)
    print("\nPOS Tags:")
    for w, t in zip(words, tags_pred):
        print(f"{w:15} --> {t}")
print("\nFinished.")

Total Sentences: 47959
Number of POS Tags: 42
Vocabulary Size: 31916
Predicting...

MODEL EVALUATION

Accuracy: 0.9440

Classification Report:

              precision    recall  f1-score   support

           $       0.96      1.00      0.98       208
           ,       0.97      1.00      0.99      6544
           .       0.94      1.00      0.97      9565
           :       1.00      0.32      0.49       167
           ;       0.95      0.49      0.65        43
          CC       1.00      1.00      1.00      4618
          CD       0.95      0.93      0.94      4983
          DT       0.96      1.00      0.98     19735
          EX       1.00      0.78      0.88       137
          IN       0.96      0.99      0.98     24247
          JJ       0.89      0.91      0.90     15524
         JJR       0.86      0.85      0.85       557
         JJS       0.93      0.87      0.90       638
         LRB       1.00      0.89      0.94       137
          MD       0.96      0.99      0.98  